In [3]:
import pandas as pd

# Load the datasets
swc_df = pd.read_csv("SWC.csv")
historical_df = pd.read_csv("HISTORICAL_Sidewalk_Caf__Licenses_and_Applications_20250408.csv")

# Print shape of each dataset
print("SWC.csv shape:", swc_df.shape)  # (rows, columns)
print("Historical dataset shape:", historical_df.shape)


SWC.csv shape: (1116, 47)
Historical dataset shape: (324, 47)


In [4]:
import pandas as pd

# Load the datasets
swc_df = pd.read_csv("SWC.csv")
historical_df = pd.read_csv("HISTORICAL_Sidewalk_Caf__Licenses_and_Applications_20250408.csv")

# Combine both datasets
combined_df = pd.concat([swc_df, historical_df], ignore_index=True)

# Drop duplicate entries based on the 'APP_ID' field
combined_df = combined_df.drop_duplicates(subset="APP_ID", keep="first")

# Save the combined dataset to a new CSV (optional)
combined_df.to_csv("Combined_Sidewalk_Cafes.csv", index=False)

print("Combined dataset created with", combined_df.shape, "unique entries.")


Combined dataset created with (1116, 47) unique entries.


In [5]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load the combined dataset
df = pd.read_csv("Combined_Sidewalk_Cafes.csv")

# ✅ Step 1: Filter for Manhattan using COMMUNITY_DISTRICT codes (101 to 112)
manhattan_df = df[df["COMMUNITY_DISTRICT"].between(101, 112)]

# ✅ Step 2: Drop rows without valid FINAL_X and FINAL_Y coordinates
manhattan_df = manhattan_df.dropna(subset=["FINAL_X", "FINAL_Y"])

# ✅ Step 3: Create geometry column using FINAL_X (X) and FINAL_Y (Y), assuming EPSG:2263
geometry = [Point(xy) for xy in zip(manhattan_df["FINAL_X"], manhattan_df["FINAL_Y"])]
gdf = gpd.GeoDataFrame(manhattan_df, geometry=geometry, crs="EPSG:2263")

# ✅ Step 4: Reproject to EPSG:4326 (WGS 84 - for web maps)
gdf = gdf.to_crs(epsg=4326)

# ✅ Step 5: Save to GeoJSON
output_path = "Manhattan_Sidewalk_Cafes.geojson"
gdf.to_file(output_path, driver="GeoJSON")

# ✅ Step 6: Summary
print(f"✅ Total sidewalk cafes in Manhattan with valid coordinates: {len(gdf)}")
print(f"🌐 GeoJSON file saved as: {output_path}")


✅ Total sidewalk cafes in Manhattan with valid coordinates: 785
🌐 GeoJSON file saved as: Manhattan_Sidewalk_Cafes.geojson


In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Load CSV
csv_path = "Combined_Sidewalk_Cafes.csv"  # Update path if needed
df = pd.read_csv(csv_path)

# Drop rows with missing coordinate data
df = df.dropna(subset=["FINAL_X", "FINAL_Y"])

# Create geometry column (assuming FINAL_X = X and FINAL_Y = Y in EPSG:2263)
geometry = [Point(xy) for xy in zip(df["FINAL_X"], df["FINAL_Y"])]

# Convert to GeoDataFrame with the original coordinate reference system
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:2263")

# Reproject to EPSG:4326 (lat/lon for web maps)
gdf = gdf.to_crs(epsg=4326)

# Save to GeoJSON
output_path = "Sidewalk_Cafes.geojson"
gdf.to_file(output_path, driver="GeoJSON")

print("GeoJSON file created:", output_path)


GeoJSON file created: Sidewalk_Cafes.geojson
